# Attempt 01

In [ ]:
# K1 — paths and settings. Peak RAM: ~0 MB
from pathlib import Path
from itertools import groupby
from collections import Counter, defaultdict
import csv, json, time

project_dir = Path(r"D:\Project\Radiology_RAG")
REF_OUT  = project_dir/"data"/"subtask_data_coverage"/"reference_data"  # filtered UMLS 2026AA reference (MRCONSO, MRSTY)
OUT      = project_dir/"data"/"subtask_data_coverage"/"filtered_data"   # English UMLS 2026AA (MRDEF, MRREL, MRHIER)
GOLD_DIR = project_dir/"data"/"subtask_gold_concepts_det"
KB_OUT   = project_dir/"data"/"subtask_kb_build"
KB_OUT.mkdir(parents=True, exist_ok=True)

# Seed branches = top-level MeSH 2026 tree categories (NLM MeSH Tree Structures).
# Reason: KB scope is set by a published hierarchy, independent of the captions (no leakage).
SEED_BRANCHES = {
    "C08": "Respiratory Tract Diseases",
    "C14": "Cardiovascular Diseases",   # heart shadow, aorta, pulmonary embolism appear on chest images
    "A04": "Respiratory System",
    "A07": "Cardiovascular System",
}

# 1-hop neighbour relations (MRREL REL column, UMLS Reference Manual).
# PAR/CHD = parent/child, RB/RN = broader/narrower, RO = other relation (e.g. finding site).
HIER_RELS  = {"PAR", "CHD", "RB", "RN"}
OTHER_RELS = {"RO"}

# Definition source order (MRDEF SAB). MSH first: seeds come from MeSH.
# NCI next: largest curated English definition set. Then CSP, MEDLINEPLUS, then any other kept source.
DEF_PRIORITY = ["MSH", "NCI", "CSP", "MEDLINEPLUS"]
MAX_SYN = 10   # shortest 10 synonyms per concept; keeps each document inside the 120-word context budget

# Bit flags stored in one bytearray CUI mask (index = numeric part of the CUI)
REF, SEED, NB_HIER, NB_RO, KB = 1, 2, 4, 8, 16
MASK_SIZE = 1 << 23            # 8.4M slots, above the highest CUI number; 8 MB of RAM
def cui_i(c): return int(c[1:])

In [ ]:
# K2 — mark every CUI in the filtered reference + collect its sources (SAB).
# Reads 511 MB, ~1-2 min. Peak RAM: ~10 MB
t0 = time.time()
mask = bytearray(MASK_SIZE)
ref_sabs = set()
n_ref, prev = 0, ""
with open(REF_OUT/"MRCONSO.RRF", encoding="utf-8") as f:
    for line in f:
        p = line.split("|", 12)          # p[0]=CUI, p[11]=SAB
        cui = p[0]
        if cui != prev:
            assert cui > prev, f"reference MRCONSO not sorted by CUI at {cui}"
            prev = cui
            mask[cui_i(cui)] |= REF
            n_ref += 1
        ref_sabs.add(p[11])
assert "MSH" in ref_sabs, "MeSH missing from the reference; seeds cannot be built"
print(f"reference CUIs: {n_ref:,} | sources kept: {len(ref_sabs)} | {time.time()-t0:.0f}s")

In [ ]:
# K3 — seeds: MeSH atoms whose tree number (MRHIER HCD) starts in a seed branch.
# Reads 3.7 GB, ~3-6 min. Peak RAM: < 50 MB (seed sets hold a few thousand CUIs)
t0 = time.time()
seeds_by_branch = defaultdict(set)
n_msh = 0
with open(OUT/"MRHIER_ENG.RRF", encoding="utf-8") as f:
    for line in f:
        if "|MSH|" not in line:          # cheap pre-check before split
            continue
        p = line.split("|")              # p[0]=CUI, p[4]=SAB, p[7]=HCD (tree number)
        if p[4] != "MSH" or not p[7]:
            continue
        n_msh += 1
        top = p[7].split(".", 1)[0]
        if top in SEED_BRANCHES and mask[cui_i(p[0])] & REF:
            seeds_by_branch[top].add(p[0])
assert n_msh > 0, "no MSH rows with tree numbers in MRHIER"

for b, name in SEED_BRANCHES.items():
    print(f"{b} {name:28s} {len(seeds_by_branch[b]):7,} CUIs")
    for c in seeds_by_branch[b]:
        mask[cui_i(c)] |= SEED
n_seed = sum(1 for m in mask if m & SEED)
assert n_seed > 0
print(f"seed CUIs (union): {n_seed:,} | MSH rows with tree number: {n_msh:,} | {time.time()-t0:.0f}s")

In [ ]:
# K4 — 1-hop neighbours of seeds in MRREL. Hierarchy and RO neighbours get separate flags,
# so K5 can show what each adds. Reads 2.9 GB, ~3-5 min. Peak RAM: ~10 MB
t0 = time.time()
NB_FLAG = {r: NB_HIER for r in HIER_RELS} | {r: NB_RO for r in OTHER_RELS}
rel_rows = Counter()
with open(OUT/"MRREL_ENG.RRF", encoding="utf-8") as f:
    for line in f:
        p = line.split("|")              # p[0]=CUI1, p[3]=REL, p[4]=CUI2, p[10]=SAB, p[14]=SUPPRESS
        flag = NB_FLAG.get(p[3])
        if flag is None or p[10] not in ref_sabs or p[14] != "N":
            continue
        a, b = cui_i(p[0]), cui_i(p[4])
        if a == b:
            continue
        ma, mb = mask[a], mask[b]
        if not (ma & REF and mb & REF):
            continue
        if ma & SEED and not mb & SEED:
            mask[b] |= flag; rel_rows[p[3]] += 1
        elif mb & SEED and not ma & SEED:
            mask[a] |= flag; rel_rows[p[3]] += 1

n_hier = sum(1 for m in mask if m & NB_HIER and not m & SEED)
n_ro_only = sum(1 for m in mask if m & NB_RO and not m & (SEED | NB_HIER))
print("seed-to-neighbour rows by REL:", dict(rel_rows))
print(f"hierarchy neighbours: {n_hier:,} | extra from RO only: {n_ro_only:,} | {time.time()-t0:.0f}s")

In [ ]:
# K5 — KB ceiling = share of gold concepts that are in the KB (upper bound on retrieval recall).
# micro = over all caption-concept pairs, macro = mean per caption, unique = over distinct CUIs.
# Peak RAM: < 20 MB
gold = []
with open(GOLD_DIR/"mlbv_chest_validation_gold.tsv", encoding="utf-8", newline="") as f:
    for r in csv.DictReader(f, delimiter="\t"):
        ga = [c for c in r["gold_all"].split(";") if c]
        gh = [c for c in r["gold_hard"].split(";") if c]
        nm = [n.strip() for n in r["gold_hard_names"].split("|")] if r["gold_hard_names"] else []
        gold.append({"id": r["id"], "all": ga, "hard": gh,
                     "hard_names": nm if len(nm) == len(gh) else [""] * len(gh)})
n_empty = sum(1 for g in gold if not g["all"])
gold = [g for g in gold if g["all"]]
print(f"captions: {len(gold)+n_empty:,} | empty gold_all (excluded): {n_empty} | used: {len(gold):,}")

outside_ref = {c for g in gold for c in g["all"] if not mask[cui_i(c)] & REF}
print(f"gold CUIs not in the reference at all: {len(outside_ref)} (no KB variant can hold these)")

VARIANTS = {
    "A0 seeds only":                        SEED,
    "A1 seeds + hierarchy neighbours":      SEED | NB_HIER,
    "A2 seeds + hierarchy + RO neighbours": SEED | NB_HIER | NB_RO,
}

def ceiling(col, flags):
    hit = tot = 0; per = []; uniq = set(); uniq_hit = set()
    for g in gold:
        cs = g[col]
        if not cs:
            continue
        hs = [c for c in cs if mask[cui_i(c)] & flags]
        hit += len(hs); tot += len(cs); per.append(len(hs) / len(cs))
        uniq.update(cs); uniq_hit.update(hs)
    return hit / tot, sum(per) / len(per), len(uniq_hit) / len(uniq)

ceiling_table = []
for name, fl in VARIANTS.items():
    row = {"variant": name, "kb_cuis": sum(1 for m in mask if m & fl)}
    for col in ("all", "hard"):
        mi, ma, un = ceiling(col, fl)
        row |= {f"{col}_micro": round(mi, 4), f"{col}_macro": round(ma, 4), f"{col}_unique": round(un, 4)}
    ceiling_table.append(row)

print(f"\n{'variant':38s} {'KB CUIs':>9s} | gold_all micro/macro/unique | gold_hard micro/macro/unique")
for r in ceiling_table:
    print(f"{r['variant']:38s} {r['kb_cuis']:9,} | "
          f"{r['all_micro']:.3f} / {r['all_macro']:.3f} / {r['all_unique']:.3f}      | "
          f"{r['hard_micro']:.3f} / {r['hard_macro']:.3f} / {r['hard_unique']:.3f}")
with open(KB_OUT/"kb_ceiling_variants.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(ceiling_table[0])); w.writeheader(); w.writerows(ceiling_table)

In [ ]:
# K6 — fix the KB variant. Default A1; switch to A2 only if K5 shows RO adds a large ceiling gain
# for a manageable KB size. Write the reason in BAB III. Safe to re-run. Peak RAM: < 10 MB
KB_VARIANT = "A1 seeds + hierarchy neighbours"
kb_flags = VARIANTS[KB_VARIANT]

n_kb = 0
for i, m in enumerate(mask):
    if m:
        new = (m & ~KB) | (KB if m & kb_flags else 0)
        mask[i] = new
        n_kb += bool(new & KB)
print(f"KB variant: {KB_VARIANT} | KB CUIs: {n_kb:,}")

missing = Counter()
for g in gold:
    for c, n in zip(g["hard"], g["hard_names"]):
        if not mask[cui_i(c)] & KB:
            missing[(c, n)] += 1
print(f"\nmissing gold_hard concepts: {len(missing):,} distinct, {sum(missing.values()):,} mentions. Top 25:")
for (c, n), k in missing.most_common(25):
    print(f"  {k:4d}  {c}  {n}")
with open(KB_OUT/"kb_missing_gold_hard.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f); w.writerow(["cui", "name", "captions"])
    for (c, n), k in missing.most_common():
        w.writerow([c, n, k])

In [ ]:
# K7 — one document per KB CUI, by a streaming merge-join of three CUI-sorted files:
# reference MRCONSO (names) + reference MRSTY (semantic types) + MRDEF_ENG (definitions).
# Reads ~700 MB, ~2-4 min. Peak RAM: ~10 MB (one CUI block at a time)
t0 = time.time()

def cui_blocks(path, keep):
    """Yield (cui, rows) from a CUI-sorted RRF file; asserts the sort order."""
    prev = ""
    with open(path, encoding="utf-8") as f:
        for cui, grp in groupby((l.rstrip("\n").split("|") for l in f), key=lambda p: p[0]):
            assert cui > prev, f"{path.name} not sorted by CUI at {cui}"
            prev = cui
            if keep(cui):
                yield cui, list(grp)

in_kb = lambda c: bool(mask[cui_i(c)] & KB)

def pick_name(rows):   # MRCONSO: TS=P, STT=PF, ISPREF=Y marks the concept's preferred string
    for r in rows:
        if r[2] == "P" and r[4] == "PF" and r[6] == "Y":
            return r[14]
    for r in rows:
        if r[6] == "Y":
            return r[14]
    return rows[0][14]

def pick_def(rows):    # MRDEF: p[4]=SAB, p[5]=DEF, p[6]=SUPPRESS
    good = [r for r in rows if r[4] in ref_sabs and r[6] == "N" and r[5].strip()]
    for sab in DEF_PRIORITY:
        for r in good:
            if r[4] == sab:
                return r[5].strip(), sab
    return (good[0][5].strip(), good[0][4]) if good else ("", "")

sty_it = cui_blocks(REF_OUT/"MRSTY.RRF", in_kb)
def_it = cui_blocks(OUT/"MRDEF_ENG.RRF", in_kb)
sty_cur, def_cur = next(sty_it, None), next(def_it, None)

n_docs = n_def = n_no_sty = 0
def_src = Counter()
with open(KB_OUT/"kb_docs.jsonl", "w", encoding="utf-8") as out:
    for cui, rows in cui_blocks(REF_OUT/"MRCONSO.RRF", in_kb):
        while sty_cur and sty_cur[0] < cui: sty_cur = next(sty_it, None)
        while def_cur and def_cur[0] < cui: def_cur = next(def_it, None)
        stys = sty_cur[1] if sty_cur and sty_cur[0] == cui else []
        defs = def_cur[1] if def_cur and def_cur[0] == cui else []

        name = pick_name(rows)
        seen, syns = {name.lower()}, []
        for s in sorted({r[14] for r in rows}, key=lambda s: (len(s), s.lower())):
            if s.lower() not in seen:
                seen.add(s.lower()); syns.append(s)
        definition, dsab = pick_def(defs)

        doc = {"cui": cui, "name": name, "synonyms": syns[:MAX_SYN],
               "semantic_types": [r[3] for r in stys], "tuis": [r[1] for r in stys],
               "definition": definition, "def_source": dsab,
               "kb_role": "seed" if mask[cui_i(cui)] & SEED else "neighbour"}
        out.write(json.dumps(doc, ensure_ascii=False) + "\n")
        n_docs += 1; n_def += bool(definition); n_no_sty += not stys; def_src[dsab or "none"] += 1

assert n_docs == n_kb, f"docs {n_docs:,} != KB CUIs {n_kb:,}"
print(f"docs: {n_docs:,} | with definition: {n_def:,} ({n_def/n_docs:.1%}) | no semantic type: {n_no_sty}")
print("definition source:", def_src.most_common(8), f"| {time.time()-t0:.0f}s")

In [ ]:
# K8 — KG edges between KB concepts, in the prototype's (child, parent, rel) format.
# MRREL REL = relation of CUI2 to CUI1, so PAR means CUI2 is the parent of CUI1.
# RO has no direction; it is stored as (lower CUI, higher CUI). The prototype walks hops undirected.
# Reads 2.9 GB, ~3-5 min. Peak RAM: up to ~300 MB (edge dict with int keys)
t0 = time.time()
EDGE_RELS = HIER_RELS | OTHER_RELS
REL_NAME = {0: "isa", 1: "broader", 2: "related"}
LOW = (1 << 23) - 1
edges = {}                     # key = child_i << 23 | parent_i -> strongest rel code
rel_seen = Counter()
with open(OUT/"MRREL_ENG.RRF", encoding="utf-8") as f:
    for line in f:
        p = line.split("|")
        rel = p[3]
        if rel not in EDGE_RELS or p[10] not in ref_sabs or p[14] != "N":
            continue
        a, b = cui_i(p[0]), cui_i(p[4])
        if a == b or not (mask[a] & KB and mask[b] & KB):
            continue
        if   rel == "PAR": child, parent, code = a, b, 0
        elif rel == "CHD": child, parent, code = b, a, 0
        elif rel == "RB":  child, parent, code = a, b, 1
        elif rel == "RN":  child, parent, code = b, a, 1
        else:              child, parent, code = min(a, b), max(a, b), 2
        k = child << 23 | parent
        if code < edges.get(k, 3):
            edges[k] = code
        rel_seen[rel] += 1

with open(KB_OUT/"kg_edges.tsv", "w", encoding="utf-8", newline="") as f:
    f.write("child\tparent\trel\n")
    for k in sorted(edges):
        f.write(f"C{k >> 23:07d}\tC{k & LOW:07d}\t{REL_NAME[edges[k]]}\n")
n_edges = len(edges)
edge_types = Counter(REL_NAME[v] for v in edges.values())
del edges
print(f"MRREL rows used: {dict(rel_seen)}")
print(f"unique edges: {n_edges:,} | by type: {dict(edge_types)} | {time.time()-t0:.0f}s")

# Attempt 02

In [ ]:
# K1 — paths and settings. Peak RAM: ~0 MB
from pathlib import Path
from itertools import groupby
from collections import Counter, defaultdict
import csv, json, time

project_dir = Path(r"D:\Project\Radiology_RAG")
REF_OUT  = project_dir/"data"/"subtask_data_coverage"/"reference_data"  # filtered UMLS 2026AA reference (MRCONSO, MRSTY)
OUT      = project_dir/"data"/"subtask_data_coverage"/"filtered_data"   # English UMLS 2026AA (MRDEF, MRREL, MRHIER)
GOLD_DIR = project_dir/"data"/"subtask_gold_concepts_det"
KB_OUT   = project_dir/"data"/"subtask_kb_build"
KB_OUT.mkdir(parents=True, exist_ok=True)

# Seed branches = MeSH 2026 tree-number prefixes, checked against desc2026.xml on 2026-11-15.
# Reason: KB scope is set by a published hierarchy, independent of the captions (no leakage).
# Run 1 used only C08/C14/A04/A07 -> ceiling 32% (gold_hard); missing mass was imaging
# procedures (35%), general findings (21%) and non-respiratory anatomy (16%).
SEED_BRANCHES = {
    "C08":         "Respiratory Tract Diseases",
    "C14":         "Cardiovascular Diseases",          # heart shadow, aorta, pulmonary embolism on chest images
    "A04":         "Respiratory System",
    "A07":         "Cardiovascular System",
    "E01.370.350": "Diagnostic Imaging",               # CT, radiography, MRI, echocardiography, angiography
    "C23":         "Pathological Conditions, Signs and Symptoms",  # lesion, nodule, infiltration, calcification
    "A01.923.761": "Thorax",                           # includes Thoracic Cavity (.800) and Thoracic Wall (.850)
}

# 1-hop neighbour relations (MRREL REL column, UMLS Reference Manual).
# PAR/CHD = parent/child, RB/RN = broader/narrower, RO = other relation (e.g. finding site).
HIER_RELS  = {"PAR", "CHD", "RB", "RN"}
OTHER_RELS = {"RO"}

# Definition source order (MRDEF SAB). MSH first: seeds come from MeSH.
# NCI next: largest curated English definition set. Then CSP, MEDLINEPLUS, then any other kept source.
DEF_PRIORITY = ["MSH", "NCI", "CSP", "MEDLINEPLUS"]
MAX_SYN = 10   # shortest 10 synonyms per concept; keeps each document inside the 120-word context budget

# Bit flags stored in one bytearray CUI mask (index = numeric part of the CUI)
REF, SEED, NB_HIER, NB_RO, KB = 1, 2, 4, 8, 16
MASK_SIZE = 1 << 23            # 8.4M slots, above the highest CUI number; 8 MB of RAM
def cui_i(c): return int(c[1:])

In [ ]:
# K2 — mark every CUI in the filtered reference + collect its sources (SAB).
# Reads 511 MB, ~1-2 min. Peak RAM: ~10 MB
t0 = time.time()
mask = bytearray(MASK_SIZE)
ref_sabs = set()
n_ref, prev = 0, ""
with open(REF_OUT/"MRCONSO.RRF", encoding="utf-8") as f:
    for line in f:
        p = line.split("|", 12)          # p[0]=CUI, p[11]=SAB
        cui = p[0]
        if cui != prev:
            assert cui > prev, f"reference MRCONSO not sorted by CUI at {cui}"
            prev = cui
            mask[cui_i(cui)] |= REF
            n_ref += 1
        ref_sabs.add(p[11])
assert "MSH" in ref_sabs, "MeSH missing from the reference; seeds cannot be built"
print(f"reference CUIs: {n_ref:,} | sources kept: {len(ref_sabs)} | {time.time()-t0:.0f}s")

In [ ]:
# K3 — seeds: MeSH atoms whose tree number (MRHIER HCD) falls under a seed branch prefix.
# Prefix match on whole tree segments, so "A01.911" matches "A01.911.xxx" but not "A01.9110".
# Reads 3.7 GB, ~3-6 min. Peak RAM: < 50 MB
t0 = time.time()
seeds_by_branch = defaultdict(set)
n_msh = 0
with open(OUT/"MRHIER_ENG.RRF", encoding="utf-8") as f:
    for line in f:
        if "|MSH|" not in line:          # cheap pre-check before split
            continue
        p = line.split("|")              # p[0]=CUI, p[4]=SAB, p[7]=HCD (tree number)
        hcd = p[7]
        if p[4] != "MSH" or not hcd:
            continue
        n_msh += 1
        if not mask[cui_i(p[0])] & REF:
            continue
        for b in SEED_BRANCHES:
            if hcd == b or hcd.startswith(b + "."):
                seeds_by_branch[b].add(p[0])
assert n_msh > 0, "no MSH rows with tree numbers in MRHIER"

for b, name in SEED_BRANCHES.items():
    print(f"{b:12s} {name:45s} {len(seeds_by_branch[b]):7,} CUIs")
empty = [b for b in SEED_BRANCHES if not seeds_by_branch[b]]
assert not empty, f"0 seeds for {empty}: tree number is probably wrong, check it in the MeSH Browser"

for cs in seeds_by_branch.values():
    for c in cs:
        mask[cui_i(c)] |= SEED
n_seed = sum(1 for m in mask if m & SEED)
print(f"seed CUIs (union): {n_seed:,} | MSH rows with tree number: {n_msh:,} | {time.time()-t0:.0f}s")

In [ ]:
# K4 — 1-hop neighbours of seeds in MRREL. Hierarchy and RO neighbours get separate flags,
# so K5 can show what each adds. Reads 2.9 GB, ~3-5 min. Peak RAM: ~10 MB
t0 = time.time()
NB_FLAG = {r: NB_HIER for r in HIER_RELS} | {r: NB_RO for r in OTHER_RELS}
rel_rows = Counter()
with open(OUT/"MRREL_ENG.RRF", encoding="utf-8") as f:
    for line in f:
        p = line.split("|")              # p[0]=CUI1, p[3]=REL, p[4]=CUI2, p[10]=SAB, p[14]=SUPPRESS
        flag = NB_FLAG.get(p[3])
        if flag is None or p[10] not in ref_sabs or p[14] != "N":
            continue
        a, b = cui_i(p[0]), cui_i(p[4])
        if a == b:
            continue
        ma, mb = mask[a], mask[b]
        if not (ma & REF and mb & REF):
            continue
        if ma & SEED and not mb & SEED:
            mask[b] |= flag; rel_rows[p[3]] += 1
        elif mb & SEED and not ma & SEED:
            mask[a] |= flag; rel_rows[p[3]] += 1

n_hier = sum(1 for m in mask if m & NB_HIER and not m & SEED)
n_ro_only = sum(1 for m in mask if m & NB_RO and not m & (SEED | NB_HIER))
print("seed-to-neighbour rows by REL:", dict(rel_rows))
print(f"hierarchy neighbours: {n_hier:,} | extra from RO only: {n_ro_only:,} | {time.time()-t0:.0f}s")

In [ ]:
# K5 — KB ceiling = share of gold concepts that are in the KB (upper bound on retrieval recall).
# micro = over all caption-concept pairs, macro = mean per caption, unique = over distinct CUIs.
# Peak RAM: < 20 MB
gold = []
with open(GOLD_DIR/"mlbv_chest_validation_gold.tsv", encoding="utf-8", newline="") as f:
    for r in csv.DictReader(f, delimiter="\t"):
        ga = [c for c in r["gold_all"].split(";") if c]
        gh = [c for c in r["gold_hard"].split(";") if c]
        nm = [n.strip() for n in r["gold_hard_names"].split("|")] if r["gold_hard_names"] else []
        gold.append({"id": r["id"], "all": ga, "hard": gh,
                     "hard_names": nm if len(nm) == len(gh) else [""] * len(gh)})
n_empty = sum(1 for g in gold if not g["all"])
gold = [g for g in gold if g["all"]]
print(f"captions: {len(gold)+n_empty:,} | empty gold_all (excluded): {n_empty} | used: {len(gold):,}")

outside_ref = {c for g in gold for c in g["all"] if not mask[cui_i(c)] & REF}
print(f"gold CUIs not in the reference at all: {len(outside_ref)} (no KB variant can hold these)")

VARIANTS = {
    "A0 seeds only":                        SEED,
    "A1 seeds + hierarchy neighbours":      SEED | NB_HIER,
    "A2 seeds + hierarchy + RO neighbours": SEED | NB_HIER | NB_RO,
}

def ceiling(col, flags):
    hit = tot = 0; per = []; uniq = set(); uniq_hit = set()
    for g in gold:
        cs = g[col]
        if not cs:
            continue
        hs = [c for c in cs if mask[cui_i(c)] & flags]
        hit += len(hs); tot += len(cs); per.append(len(hs) / len(cs))
        uniq.update(cs); uniq_hit.update(hs)
    return hit / tot, sum(per) / len(per), len(uniq_hit) / len(uniq)

ceiling_table = []
for name, fl in VARIANTS.items():
    row = {"variant": name, "kb_cuis": sum(1 for m in mask if m & fl)}
    for col in ("all", "hard"):
        mi, ma, un = ceiling(col, fl)
        row |= {f"{col}_micro": round(mi, 4), f"{col}_macro": round(ma, 4), f"{col}_unique": round(un, 4)}
    ceiling_table.append(row)

print(f"\n{'variant':38s} {'KB CUIs':>9s} | gold_all micro/macro/unique | gold_hard micro/macro/unique")
for r in ceiling_table:
    print(f"{r['variant']:38s} {r['kb_cuis']:9,} | "
          f"{r['all_micro']:.3f} / {r['all_macro']:.3f} / {r['all_unique']:.3f}      | "
          f"{r['hard_micro']:.3f} / {r['hard_macro']:.3f} / {r['hard_unique']:.3f}")
with open(KB_OUT/"kb_ceiling_variants.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(ceiling_table[0])); w.writeheader(); w.writerows(ceiling_table)

In [ ]:
# K6 — fix the KB variant. Default A1; switch to A2 only if K5 shows RO adds a large ceiling gain
# for a manageable KB size. Write the reason in BAB III. Safe to re-run. Peak RAM: < 10 MB
KB_VARIANT = "A1 seeds + hierarchy neighbours"
kb_flags = VARIANTS[KB_VARIANT]

n_kb = 0
for i, m in enumerate(mask):
    if m:
        new = (m & ~KB) | (KB if m & kb_flags else 0)
        mask[i] = new
        n_kb += bool(new & KB)
print(f"KB variant: {KB_VARIANT} | KB CUIs: {n_kb:,}")

missing = Counter()
for g in gold:
    for c, n in zip(g["hard"], g["hard_names"]):
        if not mask[cui_i(c)] & KB:
            missing[(c, n)] += 1
print(f"\nmissing gold_hard concepts: {len(missing):,} distinct, {sum(missing.values()):,} mentions. Top 25:")
for (c, n), k in missing.most_common(25):
    print(f"  {k:4d}  {c}  {n}")
with open(KB_OUT/"kb_missing_gold_hard.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f); w.writerow(["cui", "name", "captions"])
    for (c, n), k in missing.most_common():
        w.writerow([c, n, k])

In [ ]:
# K7 — one document per KB CUI, by a streaming merge-join of three CUI-sorted files:
# reference MRCONSO (names) + reference MRSTY (semantic types) + MRDEF_ENG (definitions).
# Reads ~700 MB, ~2-4 min. Peak RAM: ~10 MB (one CUI block at a time)
t0 = time.time()

def cui_blocks(path, keep):
    """Yield (cui, rows) from a CUI-sorted RRF file; asserts the sort order."""
    prev = ""
    with open(path, encoding="utf-8") as f:
        for cui, grp in groupby((l.rstrip("\n").split("|") for l in f), key=lambda p: p[0]):
            assert cui > prev, f"{path.name} not sorted by CUI at {cui}"
            prev = cui
            if keep(cui):
                yield cui, list(grp)

in_kb = lambda c: bool(mask[cui_i(c)] & KB)

def pick_name(rows):   # MRCONSO: TS=P, STT=PF, ISPREF=Y marks the concept's preferred string
    for r in rows:
        if r[2] == "P" and r[4] == "PF" and r[6] == "Y":
            return r[14]
    for r in rows:
        if r[6] == "Y":
            return r[14]
    return rows[0][14]

def pick_def(rows):    # MRDEF: p[4]=SAB, p[5]=DEF, p[6]=SUPPRESS
    good = [r for r in rows if r[4] in ref_sabs and r[6] == "N" and r[5].strip()]
    for sab in DEF_PRIORITY:
        for r in good:
            if r[4] == sab:
                return r[5].strip(), sab
    return (good[0][5].strip(), good[0][4]) if good else ("", "")

sty_it = cui_blocks(REF_OUT/"MRSTY.RRF", in_kb)
def_it = cui_blocks(OUT/"MRDEF_ENG.RRF", in_kb)
sty_cur, def_cur = next(sty_it, None), next(def_it, None)

n_docs = n_def = n_no_sty = 0
def_src = Counter()
with open(KB_OUT/"kb_docs.jsonl", "w", encoding="utf-8") as out:
    for cui, rows in cui_blocks(REF_OUT/"MRCONSO.RRF", in_kb):
        while sty_cur and sty_cur[0] < cui: sty_cur = next(sty_it, None)
        while def_cur and def_cur[0] < cui: def_cur = next(def_it, None)
        stys = sty_cur[1] if sty_cur and sty_cur[0] == cui else []
        defs = def_cur[1] if def_cur and def_cur[0] == cui else []

        name = pick_name(rows)
        seen, syns = {name.lower()}, []
        for s in sorted({r[14] for r in rows}, key=lambda s: (len(s), s.lower())):
            if s.lower() not in seen:
                seen.add(s.lower()); syns.append(s)
        definition, dsab = pick_def(defs)

        doc = {"cui": cui, "name": name, "synonyms": syns[:MAX_SYN],
               "semantic_types": [r[3] for r in stys], "tuis": [r[1] for r in stys],
               "definition": definition, "def_source": dsab,
               "kb_role": "seed" if mask[cui_i(cui)] & SEED else "neighbour"}
        out.write(json.dumps(doc, ensure_ascii=False) + "\n")
        n_docs += 1; n_def += bool(definition); n_no_sty += not stys; def_src[dsab or "none"] += 1

assert n_docs == n_kb, f"docs {n_docs:,} != KB CUIs {n_kb:,}"
print(f"docs: {n_docs:,} | with definition: {n_def:,} ({n_def/n_docs:.1%}) | no semantic type: {n_no_sty}")
print("definition source:", def_src.most_common(8), f"| {time.time()-t0:.0f}s")

In [ ]:
# K8 — KG edges between KB concepts, in the prototype's (child, parent, rel) format.
# MRREL REL = relation of CUI2 to CUI1, so PAR means CUI2 is the parent of CUI1.
# RO has no direction; it is stored as (lower CUI, higher CUI). The prototype walks hops undirected.
# Reads 2.9 GB, ~3-5 min. Peak RAM: up to ~300 MB (edge dict with int keys)
t0 = time.time()
EDGE_RELS = HIER_RELS | OTHER_RELS
REL_NAME = {0: "isa", 1: "broader", 2: "related"}
LOW = (1 << 23) - 1
edges = {}                     # key = child_i << 23 | parent_i -> strongest rel code
rel_seen = Counter()
with open(OUT/"MRREL_ENG.RRF", encoding="utf-8") as f:
    for line in f:
        p = line.split("|")
        rel = p[3]
        if rel not in EDGE_RELS or p[10] not in ref_sabs or p[14] != "N":
            continue
        a, b = cui_i(p[0]), cui_i(p[4])
        if a == b or not (mask[a] & KB and mask[b] & KB):
            continue
        if   rel == "PAR": child, parent, code = a, b, 0
        elif rel == "CHD": child, parent, code = b, a, 0
        elif rel == "RB":  child, parent, code = a, b, 1
        elif rel == "RN":  child, parent, code = b, a, 1
        else:              child, parent, code = min(a, b), max(a, b), 2
        k = child << 23 | parent
        if code < edges.get(k, 3):
            edges[k] = code
        rel_seen[rel] += 1

with open(KB_OUT/"kg_edges.tsv", "w", encoding="utf-8", newline="") as f:
    f.write("child\tparent\trel\n")
    for k in sorted(edges):
        f.write(f"C{k >> 23:07d}\tC{k & LOW:07d}\t{REL_NAME[edges[k]]}\n")
n_edges = len(edges)
edge_types = Counter(REL_NAME[v] for v in edges.values())
del edges
print(f"MRREL rows used: {dict(rel_seen)}")
print(f"unique edges: {n_edges:,} | by type: {dict(edge_types)} | {time.time()-t0:.0f}s")

In [ ]:
# K9 — check every edge endpoint has a document, then save a manifest for the thesis.
# Peak RAM: ~10 MB per 50k KB CUIs
doc_cuis = set()
with open(KB_OUT/"kb_docs.jsonl", encoding="utf-8") as f:
    for line in f:
        doc_cuis.add(json.loads(line)["cui"])
assert len(doc_cuis) == n_kb

bad = n_e = 0
with open(KB_OUT/"kg_edges.tsv", encoding="utf-8") as f:
    next(f)
    for line in f:
        c, par, _ = line.rstrip("\n").split("\t")
        n_e += 1
        bad += (c not in doc_cuis) + (par not in doc_cuis)
assert bad == 0 and n_e == n_edges, f"bad endpoints: {bad}, edges {n_e} vs {n_edges}"

chosen = next(r for r in ceiling_table if r["variant"] == KB_VARIANT)
manifest = {
    "umls_release": "2026AA", "seed_branches": SEED_BRANCHES, "kb_variant": KB_VARIANT,
    "neighbour_rels": sorted(HIER_RELS | (OTHER_RELS if kb_flags & NB_RO else set())),
    "edge_rels": sorted(EDGE_RELS), "def_priority": DEF_PRIORITY, "max_synonyms": MAX_SYN,
    "kb_cuis": n_kb, "seed_cuis": n_seed, "docs_with_definition": n_def, "kg_edges": n_edges,
    "edge_types": dict(edge_types), "ceiling": chosen,
    "gold_file": str(GOLD_DIR/"mlbv_chest_validation_gold.tsv"), "built": time.strftime("%Y-%m-%d %H:%M"),
}
(KB_OUT/"kb_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(manifest, indent=2, ensure_ascii=False))
print("\nAll checks passed. Restart the kernel now.")